# Restaurant and Dish Candidate Filtering

This notebook filters real restaurant dishes using the user's city, locality, budget, dietary preference, and predicted cuisines. The returned dishes still pass through RAG and the safety rule engine before recommendation.

In [ ]:
from pathlib import Path
import pandas as pd

## 1. Load the restaurant dataset

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "swiggy_cleaned_sample_expanded.csv"

required_columns = [
    "restaurant_id", "restaurant_name", "city", "locality", "address",
    "cuisine", "menu_category", "dish_name", "dish_price", "veg_nonveg",
    "restaurant_rating", "rating_count",
]

dishes = pd.read_csv(DATA_PATH, usecols=required_columns, low_memory=False)

## 2. Prepare the data

In [ ]:
def text_key(value):
    return " ".join(str(value).strip().casefold().split())


def cuisine_key(value):
    if pd.isna(value):
        return "|"
    labels = [text_key(label) for label in str(value).split(",")]
    return "|" + "|".join(labels) + "|"

In [ ]:
dishes_clean = dishes.copy()
dishes_clean["_city_key"] = dishes_clean["city"].map(text_key)
dishes_clean["_locality_key"] = dishes_clean["locality"].map(text_key)
dishes_clean["_dietary_key"] = dishes_clean["veg_nonveg"].map(text_key)
dishes_clean["_cuisine_key"] = dishes_clean["cuisine"].map(cuisine_key)

## 3. Prepare user and classifier inputs

In [ ]:
def dietary_key(preference):
    return {
        "Vegetarian": "veg",
        "Non-Vegetarian": "non-veg",
    }.get(preference)


def ranked_cuisines(predictions):
    if isinstance(predictions, str):
        return [predictions]

    return [
        prediction["cuisine"] if isinstance(prediction, dict) else prediction
        for prediction in predictions
    ]

## 4. Filter and rank dishes

In [ ]:
def apply_hard_constraints(frame, city, budget, preference, minimum_rating):
    candidates = frame[
        (frame["_city_key"] == text_key(city))
        & (frame["dish_price"] <= budget)
    ]

    if preference is not None:
        candidates = candidates[candidates["_dietary_key"] == preference]
    if minimum_rating is not None:
        candidates = candidates[candidates["restaurant_rating"] >= minimum_rating]

    return candidates

In [ ]:
def find_candidate_match(base_candidates, cuisines, locality, allow_citywide_fallback):
    for cuisine in cuisines:
        cuisine_token = f"|{text_key(cuisine)}|"
        cuisine_candidates = base_candidates[
            base_candidates["_cuisine_key"].str.contains(cuisine_token, regex=False)
        ]

        if locality:
            local_candidates = cuisine_candidates[
                cuisine_candidates["_locality_key"] == locality
            ]
            if not local_candidates.empty:
                return local_candidates

        if (allow_citywide_fallback or not locality) and not cuisine_candidates.empty:
            return cuisine_candidates

    return base_candidates.iloc[0:0]

In [ ]:
def rank_candidates(frame, top_n):
    ranked = frame.sort_values(
        by=["restaurant_rating", "rating_count", "dish_price"],
        ascending=[False, False, True],
        na_position="last",
    ).head(top_n)

    result = ranked[required_columns].reset_index(drop=True).copy()
    result.insert(0, "rank", range(1, len(result) + 1))
    return result

In [ ]:
def filter_dish_candidates(
    frame,
    user_city,
    user_locality,
    predicted_cuisines,
    user_budget,
    dietary_preference=None,
    minimum_rating=None,
    top_n=10,
    allow_citywide_fallback=True,
):
    budget = float(user_budget)
    cuisines = ranked_cuisines(predicted_cuisines)
    preference = dietary_key(dietary_preference)
    locality = text_key(user_locality) if user_locality else ""

    candidates = apply_hard_constraints(
        frame, user_city, budget, preference, minimum_rating
    )
    candidates = find_candidate_match(
        candidates, cuisines, locality, allow_citywide_fallback
    )
    return rank_candidates(candidates, top_n)

## 5. Example

In [ ]:
candidates = filter_dish_candidates(
    dishes_clean,
    user_city="Bengaluru",
    user_locality="HSR",
    predicted_cuisines=[
        {"cuisine": "North Indian", "probability": 0.78},
        {"cuisine": "Chinese", "probability": 0.16},
    ],
    dietary_preference="Vegetarian",
    user_budget=250,
    minimum_rating=4.0,
    top_n=10,
)
candidates

## Next Step

These candidates are real dataset rows, but they are not yet safety-approved. RAG retrieves ingredient knowledge and the safety rule engine checks dietary and allergy conflicts before the LLM prepares the final explanation.